# Qwen3.5-0.8B LoRA Fine-Tuning — Colab (T4 x1)

Fine-tune Qwen3.5-0.8B on one or more datasets published under
`michaelowusuntim6` on Hugging Face.

**Hardware (Colab free tier):** T4 x1, 16 GB VRAM, bf16 supported.
**Estimated 1 epoch on 10k records: ~15–25 minutes.**

Set `DATASETS` below to a single dataset, a list (concatenate), or a
dict with weights (interleave — recommended for multi-dataset).
Set `PUSH_TO_HUB = True` to upload the trained adapter to Hugging
Face as `michaelowusuntim6/<expert-name>-lora`.

Session limits: Colab free tier disconnects after ~90 minutes idle
and caps sessions at ~12 hours. Keep the tab active or mount Drive
and checkpoint there.


In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps transformers trl peft accelerate bitsandbytes datasets
!pip install -q --no-deps unsloth_zoo


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, concatenate_datasets, interleave_datasets
from trl import SFTTrainer
from transformers import TrainingArguments

# ── CONFIG ──────────────────────────────────────
# Single dataset:
#   DATASETS = "code-review-qwen35"
# Multiple datasets, concatenated:
#   DATASETS = ["code-review-qwen35", "debug-qwen35"]
# Multiple datasets, weighted mix (recommended):
DATASETS = {
    "code-review-qwen35": 0.6,
    "debug-qwen35": 0.4,
}

EXPERT_NAME = "code-review-lora"   # HF repo suffix, no -qwen35
MAX_SEQ_LEN = 8192
PUSH_TO_HUB = False                # set True to upload adapter
OUT_DIR = "/content"
SEED = 3407

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())


In [ ]:
if isinstance(DATASETS, str):
    ds = load_dataset(f"michaelowusuntim6/{DATASETS}", split="train")
    print(f"Single dataset: {len(ds)} records")
elif isinstance(DATASETS, list):
    splits = [load_dataset(f"michaelowusuntim6/{n}", split="train") for n in DATASETS]
    ds = concatenate_datasets(splits)
    print(f"Concatenated {len(DATASETS)} datasets: {len(ds)} records")
elif isinstance(DATASETS, dict):
    splits = [load_dataset(f"michaelowusuntim6/{n}", split="train") for n in DATASETS]
    weights = list(DATASETS.values())
    assert abs(sum(weights) - 1.0) < 1e-6, f"Weights must sum to 1.0, got {sum(weights)}"
    ds = interleave_datasets(splits, probabilities=weights, seed=SEED)
    print(f"Interleaved {len(DATASETS)} datasets: {len(ds)} records")
else:
    raise ValueError("DATASETS must be str, list, or dict")

# Sanity: content must be a string for FastLanguageModel
sample = ds[0]["messages"][0]
print(f"First record content type: {type(sample['content']).__name__}")
assert isinstance(sample["content"], str), "content must be str, not list"


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-0.8B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto -> bf16 on T4
    load_in_4bit=False,    # bf16 LoRA; QLoRA not recommended for Qwen3.5
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


In [ ]:
def format_record(ex):
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return ex

ds = ds.map(format_record, num_proc=2)
print(len(ds), "formatted records")


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,        # CRITICAL - do not omit
    dataset_num_proc=2,
    packing=False,                      # keep False for long sequences
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=False,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        output_dir=f"{OUT_DIR}/outputs/{EXPERT_NAME}",
        save_strategy="epoch",
        save_total_limit=1,
        seed=SEED,
        report_to="none",
    ),
)
trainer.train()


In [ ]:
model.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
tokenizer.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print("Adapter saved to", f"{OUT_DIR}/adapters/{EXPERT_NAME}")

# Optional: mount Drive to survive an idle disconnect
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r {OUT_DIR}/adapters/{EXPERT_NAME} /content/drive/MyDrive/


In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import login
    import os
    login(token=os.environ["HF_TOKEN"])
    model.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}")
    tokenizer.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}")
    print(f"Pushed: https://huggingface.co/michaelowusuntim6/{EXPERT_NAME}")
else:
    print("PUSH_TO_HUB is False — set to True to upload")


In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "How do I fix a memory leak in C?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
